In [1]:
%pwd

'/Users/john/code/ace/c477-dbt/notebooks'

In [2]:
%load_ext jupyter_black

In [14]:
from pathlib import Path

import pandas as pd
from rapidfuzz import process
from tqdm import tqdm

ROOT = Path("..")

## mart_address_profiling / os__properties - UPRN matching

In [10]:
epc = pd.read_csv(
    ROOT / "data" / "interim" / "mart_address_profiling.csv",
    low_memory=False,
)
os = pd.read_csv(ROOT / "data" / "raw" / "stg_os_places.csv", low_memory=False)

In [11]:
epc[epc["UPRN"].isnull()][["UPRN", "Address", "PostCode"]]

,UPRN,Address,PostCode
10,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY
57,NaN,5A Mill Hill Road,PO31 7EA
60,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE
62,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE
63,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ
...,...,...,...
66130,NaN,Basement Flat 21-22 Cross Street,PO33 2AA
66144,NaN,"Whitfields P2, The West Bay Club",PO41 0RJ
66163,NaN,"Ground Floor Flat, Nodes Point Holiday Park, N...",PO33 1YA
66227,NaN,"Curlew Cottage, Salterns Village, Salterns Road",PO34 5AQ


In [12]:
epc[epc["UPRN"].isna()]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
10,NaN,NaN,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY,56,NaN,NaN,NaN,4.2,...,OtherPremisesBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,94,62,94.0
57,NaN,NaN,NaN,5A Mill Hill Road,PO31 7EA,72,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,89,47,59.0
60,NaN,NaN,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE,73,NaN,NaN,NaN,2.0,...,Other,NaN,NaN,NaN,0.0,0.0,NaN,95,54,60.0
62,NaN,NaN,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.0
63,NaN,NaN,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ,66,NaN,NaN,NaN,3.2,...,Solid,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,199,90,89.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66130,NaN,NaN,NaN,Basement Flat 21-22 Cross Street,PO33 2AA,58,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazingUnknownDate,0.0,0.0,Natural,105,152,146.0
66144,NaN,NaN,NaN,"Whitfields P2, The West Bay Club",PO41 0RJ,76,NaN,NaN,NaN,2.2,...,Suspended,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,215,191,85.0
66163,NaN,NaN,NaN,"Ground Floor Flat, Nodes Point Holiday Park, N...",PO33 1YA,37,NaN,NaN,NaN,3.6,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,536,65,36.0
66227,NaN,NaN,NaN,"Curlew Cottage, Salterns Village, Salterns Road",PO34 5AQ,47,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,825,156,104.0


In [13]:
os[["uprn", "address", "postcode"]]

,uprn,address,postcode
0,10003317367,"BISCOES SOLICITORS, 143, HIGH STREET, NEWPORT,...",PO30 1TY
1,10003323356,"8, SILVER BIRCH DRIVE, NEWPORT, PO30 5AG",PO30 5AG
2,100060752724,"27, ATKINSON DRIVE, NEWPORT, PO30 2LJ",PO30 2LJ
3,100062422315,"1 SHEAT COTTAGES, BROOK LANE, CHILLERTON, NEWP...",PO30 3EW
4,100062423020,"5 TENNYSON VIEW, ELM LANE, CALBOURNE, NEWPORT,...",PO30 4JS
...,...,...,...
90771,100060775880,"ISLAND COTTAGE, TENNYSON ROAD, YARMOUTH, PO41 0PX",PO41 0PX
90772,100062448970,"FORT VICTORIA REPTILARIUM, FORT VICTORIA COUNT...",PO41 0RR
90773,10094234389,"9, BOULDNOR MEAD, BOULDNOR, YARMOUTH, PO41 0LA",PO41 0LA
90774,10094232487,"LEE BARN, LEE FARM, MAIN ROAD, WELLOW, YARMOUT...",PO41 0SY


In [15]:
# Sample data (Replace these with your actual DataFrames)
epc_data = epc[epc["UPRN"].isna()].head(50).copy()
os_data = os.copy()

epc_df = pd.DataFrame(epc_data)
os_df = pd.DataFrame(os_data)


# Function to get the closest match and its UPRN based on the address and postcode
def match_address(address, postcode, choices_df):
    combined_string = address + " " + postcode
    # Using the `extractOne` method to get the best match
    best_match = process.extractOne(
        combined_string,
        choices_df["address"] + " " + choices_df["postcode"],
    )

    # Optional: Set a threshold for a match, e.g. 80
    if best_match[1] > 80:
        matched_uprn = choices_df.loc[
            choices_df["address"] + " " + choices_df["postcode"] == best_match[0],
            "uprn",
        ].iloc[0]
        return matched_uprn
    else:
        return None


# Using tqdm in the apply function for a progress bar
tqdm.pandas()
epc_df["UPRN"] = epc_df.progress_apply(
    lambda row: match_address(row["Address"], row["PostCode"], os_df),
    axis=1,
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:09<00:00,  5.02it/s]


In [16]:
epc_df

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
10,100062624793,NaN,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY,56,NaN,NaN,NaN,4.2,...,OtherPremisesBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,94,62,94.00
57,100060749924,NaN,NaN,5A Mill Hill Road,PO31 7EA,72,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,89,47,59.00
60,10024249819,NaN,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE,73,NaN,NaN,NaN,2.0,...,Other,NaN,NaN,NaN,0.0,0.0,NaN,95,54,60.00
62,10091028677,NaN,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.00
63,100062442066,NaN,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ,66,NaN,NaN,NaN,3.2,...,Solid,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,199,90,89.00
83,10094231512,NaN,NaN,"Flat 3, Building 33, Oak Vale",PO33 1FP,84,NaN,NaN,NaN,1.3,...,Other,NaN,NaN,NaN,NaN,0.0,NaN,77,49,64.12
118,100062442066,NaN,NaN,"The Hilliers, Pritchetts Way, Rookley",PO38 3LT,74,NaN,NaN,NaN,2.5,...,Other,NaN,NaN,NaN,0.0,NaN,NaN,293,69,111.00
141,10003317367,NaN,NaN,"Flat 1, 3 Bowling Green Lane",PO30 1RR,61,NaN,NaN,NaN,2.8,...,OtherPremisesBelow,NaN,NaN,NaN,NaN,0.0,NaN,105,25,49.09
177,100062422315,NaN,NaN,"Flat 12, Harbour View, Fairlee Road",PO30 2EA,81,NaN,NaN,NaN,1.4,...,Other,NaN,NaN,NaN,NaN,0.0,NaN,77,41,56.59
192,100060747579,NaN,NaN,"3 Woodlands Landguard Holiday Park, Landguard ...",PO37 7PJ,40,NaN,NaN,NaN,5.0,...,Suspended,NoInsulation,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,208,43,59.95


---

In [19]:
# Sample data (Replace these with your actual DataFrames)
epc_data = epc[epc["UPRN"].isna()].copy()
os_data = os.copy()

epc_df = pd.DataFrame(epc_data)
os_df = pd.DataFrame(os_data)


# Function to get the closest match and its UPRN based on the address and postcode
def match_address(address, postcode, choices_df):
    combined_string = address + " " + postcode
    # Using the `extractOne` method to get the best match
    best_match = process.extractOne(
        combined_string,
        choices_df["address"] + " " + choices_df["postcode"],
    )

    # Optional: Set a threshold for a match, e.g. 80
    if best_match[1] > 80:
        matched_uprn = choices_df.loc[
            choices_df["address"] + " " + choices_df["postcode"] == best_match[0],
            "uprn",
        ].iloc[0]
        return matched_uprn
    else:
        return None


# Using tqdm in the apply function for a progress bar
tqdm.pandas()
epc_df["UPRN"] = epc_df.progress_apply(
    lambda row: match_address(row["Address"], row["PostCode"], os_df),
    axis=1,
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1610/1610 [05:07<00:00,  5.24it/s]


In [20]:
epc_df

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
10,1.000626e+11,NaN,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY,56,NaN,NaN,NaN,4.2,...,OtherPremisesBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,94,62,94.0
57,1.000607e+11,NaN,NaN,5A Mill Hill Road,PO31 7EA,72,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,89,47,59.0
60,1.002425e+10,NaN,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE,73,NaN,NaN,NaN,2.0,...,Other,NaN,NaN,NaN,0.0,0.0,NaN,95,54,60.0
62,1.009103e+10,NaN,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.0
63,1.000624e+11,NaN,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ,66,NaN,NaN,NaN,3.2,...,Solid,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,199,90,89.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66130,1.002425e+10,NaN,NaN,Basement Flat 21-22 Cross Street,PO33 2AA,58,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazingUnknownDate,0.0,0.0,Natural,105,152,146.0
66144,1.000608e+11,NaN,NaN,"Whitfields P2, The West Bay Club",PO41 0RJ,76,NaN,NaN,NaN,2.2,...,Suspended,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,215,191,85.0
66163,1.000608e+11,NaN,NaN,"Ground Floor Flat, Nodes Point Holiday Park, N...",PO33 1YA,37,NaN,NaN,NaN,3.6,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,536,65,36.0
66227,NaN,NaN,NaN,"Curlew Cottage, Salterns Village, Salterns Road",PO34 5AQ,47,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,825,156,104.0


---

In [ ]:
# Sample data (Replace these with your actual DataFrames)
epc_data = epc[epc["UPRN"].isna()].copy()
os_data = os.copy()

epc_df = pd.DataFrame(epc_data)
os_df = pd.DataFrame(os_data)


# Function to get the closest match and its UPRN based on the address and postcode
def match_address(address, postcode, choices_df):
    combined_string = address + " " + postcode
    best_match = process.extractOne(
        combined_string,
        choices_df["address"] + " " + choices_df["postcode"],
        scorer=fuzz.token_sort_ratio,
    )

    if best_match[1] > 10:
        matched_uprn = choices_df.loc[
            choices_df["address"] + " " + choices_df["postcode"] == best_match[0],
            "uprn",
        ].iloc[0]
        return matched_uprn
    else:
        return None


# Using tqdm in the apply function for a progress bar
tqdm.pandas()
epc_df["UPRN"] = epc_df.progress_apply(
    lambda row: match_address(row["Address"], row["PostCode"], os_df),
    axis=1,
)

In [22]:
epc_df["UPRN"] = epc_df["UPRN"].astype(str)

In [23]:
epc_df[epc_df["UPRN"].isna()][["UPRN", "Address", "PostCode"]]

,UPRN,Address,PostCode


In [24]:
os.query("postcode == 'PO30 1LY'")[["uprn", "udprn", "postcode", "address"]]

,uprn,udprn,postcode,address
551,10024247571,19454362,PO30 1LY,"FLAT 1, 3, ST. JOHNS ROAD, NEWPORT, PO30 1LY"
3012,10024247574,19454364,PO30 1LY,"FLAT 3, 3, ST. JOHNS ROAD, NEWPORT, PO30 1LY"
12530,10024247575,19454365,PO30 1LY,"FLAT 4, 3, ST. JOHNS ROAD, NEWPORT, PO30 1LY"
12734,10024247573,19454363,PO30 1LY,"FLAT 2, 3, ST. JOHNS ROAD, NEWPORT, PO30 1LY"


---

## mart_address_profiling

In [26]:
final_profiling = pd.read_csv(
    ROOT / "data" / "interim" / "mart_address_profiling.csv",
    low_memory=False,
)

In [27]:
final_profiling = final_profiling[~final_profiling["UPRN"].isna()]

In [28]:
final_profiling.columns

Index(['UPRN', 'UDPRN', 'ParityAddressId', 'Address', 'PostCode',
       'Environmental Impact Rating', 'Environmental Impact Rating Band',
       'Fuel Bills (£/yr)', 'Realistic Fuel Bill (Regional)', 'tCO2',
       'Heating Cost (£/yr)', 'SAP Rating', 'SAP Band', 'Lodgement Date',
       'SAP version', 'PropertyType', 'ConstructionAgeBand', 'BuiltForm',
       'StoreysCount', 'FlatLocation', 'FlatLevel', 'LowestFloorArea',
       'MainHeatingCategory', 'MainFuelType', 'MainHeatingControl',
       'DerivedSAPMainHeatingCode', 'HeatEmitterType', 'WaterHeatingFuel',
       'RoofConstruction', 'RoofInsulationLocation', 'RoofInsulationThickness',
       'WallConstruction', 'WallInsulationType', 'WallInsulationThickness',
       'FloorConstruction', 'FloorInsulation', 'FloorInsulationThickness',
       'MultipleGlazingType', 'OpenFireplacesCount', 'Renewables',
       'Ventilation', 'Water Heating Cost (£/yr)', 'Lighting Cost (£/yr)',
       'Total Floor Area'],
      dtype='object')

In [29]:
[
    "UPRN",
    "UDPRN",
    "ParityAddressId",
    "Address",
    "PostCode",
    "Environmental Impact Rating",
    "Environmental Impact Rating Band",
    "Fuel Bills (£/yr)",
    "Realistic Fuel Bill (Regional)",
    "tCO2",
    "Heating Cost (£/yr)",
    "Heating Cost (£/yr).1",
    "SAP Rating",
    "SAP Band",
    "Lodgement Date",
    "SAP version",
    "PropertyType",
    "ConstructionAgeBand",
    "BuiltForm",
    "StoreysCount",
    "FlatLocation",
    "FlatLevel",
    "LowestFloorArea",
    "MainHeatingCategory",
    "MainFuelType",
    "MainHeatingControl",
    "DerivedSAPMainHeatingCode",
    "HeatEmitterType",
    "WaterHeatingFuel",
    "RoofConstruction",
    "RoofInsulationLocation",
    "RoofInsulationThickness",
    "WallConstruction",
    "WallInsulationType",
    "WallInsulationThickness",
    "FloorConstruction",
    "FloorInsulation",
    "FloorInsulationThickness",
    "MultipleGlazingType",
    "OpenFireplacesCount",
    "Renewables",
    "Ventilation",
    "Water Heating Cost (£/yr)",
    "Lighting Cost (£/yr)",
    "Total Floor Area",
]

['UPRN',
 'UDPRN',
 'ParityAddressId',
 'Address',
 'PostCode',
 'Environmental Impact Rating',
 'Environmental Impact Rating Band',
 'Fuel Bills (£/yr)',
 'Realistic Fuel Bill (Regional)',
 'tCO2',
 'Heating Cost (£/yr)',
 'Heating Cost (£/yr).1',
 'SAP Rating',
 'SAP Band',
 'Lodgement Date',
 'SAP version',
 'PropertyType',
 'ConstructionAgeBand',
 'BuiltForm',
 'StoreysCount',
 'FlatLocation',
 'FlatLevel',
 'LowestFloorArea',
 'MainHeatingCategory',
 'MainFuelType',
 'MainHeatingControl',
 'DerivedSAPMainHeatingCode',
 'HeatEmitterType',
 'WaterHeatingFuel',
 'RoofConstruction',
 'RoofInsulationLocation',
 'RoofInsulationThickness',
 'WallConstruction',
 'WallInsulationType',
 'WallInsulationThickness',
 'FloorConstruction',
 'FloorInsulation',
 'FloorInsulationThickness',
 'MultipleGlazingType',
 'OpenFireplacesCount',
 'Renewables',
 'Ventilation',
 'Water Heating Cost (£/yr)',
 'Lighting Cost (£/yr)',
 'Total Floor Area']

In [30]:
final_profiling = final_profiling.rename(
    columns={
        "Environmental Impact Rating": "EnvironmentalImpactRating",
        "Environmental Impact Rating Band": "EnvironmentalImpactRatingBand",
        "Fuel Bills (£/yr)": "FuelBills(£/yr)",
        "Realistic Fuel Bill (Regional)": "RealisticFuelBill(Regional)",
        "Heating Cost (£/yr)": "HeatingCost(£/yr)",
        "Heating Cost (£/yr).1": "HeatingCost(£/yr).1",
        "SAP Rating": "SAPRating",
        "SAP Band": "SAPBand",
        "Lodgement Date": "LodgementDate",
        "SAP version": "SAPVersion",
        "Water Heating Cost (£/yr)": "WaterHeatingCost(£/yr)",
        "Lighting Cost (£/yr)": "LightingCost(£/yr)",
        "Total Floor Area": "TotalFloorArea",
    },
)

In [31]:
final_profiling.to_csv(ROOT / "data" / "interim" / "address_profiling.csv")

In [32]:
final_profiling

,UPRN,UDPRN,ParityAddressId,Address,PostCode,EnvironmentalImpactRating,EnvironmentalImpactRatingBand,FuelBills(£/yr),RealisticFuelBill(Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,WaterHeatingCost(£/yr),LightingCost(£/yr),TotalFloorArea
0,1.000608e+11,NaN,NaN,"30, Pitt Street",PO33 3EB,43,NaN,NaN,NaN,7.8,...,Suspended,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,157,88,126.08
1,1.000608e+11,NaN,NaN,24 Upper Highland Road,PO33 1DZ,10,NaN,NaN,NaN,9.5,...,Suspended,NoInsulation,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,456,80,68.00
2,1.000608e+11,NaN,NaN,"66, Slade Road",PO33 1EG,63,NaN,NaN,NaN,3.0,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,202,57,82.00
3,1.000608e+11,NaN,NaN,"67, Mountfield Road, Wroxall",PO38 3BX,60,NaN,NaN,NaN,3.4,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,100,46,76.00
4,1.000624e+11,NaN,NaN,"Flat 40, Homebray House, Mary Rose Avenue",PO33 4LW,69,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,319,46,49.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66302,1.002425e+10,NaN,NaN,"2 Heath Villas, Colwell Lane",PO40 9LT,51,NaN,NaN,NaN,4.2,...,Suspended,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,207,156,83.00
66303,1.000624e+11,NaN,NaN,"Nirvana, Merstone Lane, Merstone",PO30 3DF,60,NaN,NaN,NaN,3.8,...,Suspended,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,295,167,100.00
66304,1.000624e+11,NaN,NaN,"Gilmour, Elliston Road",PO39 0BA,46,NaN,NaN,NaN,4.9,...,Suspended,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,118,116,86.00
66305,1.009423e+10,NaN,NaN,8 Breakwater Way,PO36 8FE,88,NaN,NaN,NaN,1.6,...,Other,NaN,NaN,NaN,0.0,0.0,NaN,256,184,148.00


---

## mart_address_base_plus

In [ ]:
final_base_plus = pd.read_csv(
    ROOT / "data" / "interim" / "mart_address_base_plus.csv",
    low_memory=False,
)

In [34]:
final_base_plus

,UPRN,UDPRN,ADDRESS,POSTCODE_LOCATOR,X_COORDINATE,Y_COORDINATE,LATITUDE,LONGITUDE,PARENT_UPRN,WARD_CODE,LSOA11_CODE,COA11_CODE,PARISH_CODE,CONSERVATION_AREA,NATIONAL_PARK,WORLD_HERITAGE_SITE,AONB,CLASS,CLASS_DESCRIPTION
0,10003317367,54801750,"BISCOES SOLICITORS, 143, HIGH STREET, NEWPORT,...",PO30 1TY,450071.00,89235.00,50.700609,-1.292363,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CO01,Office / Work Studio
1,10003323356,19461404,"8, SILVER BIRCH DRIVE, NEWPORT, PO30 5AG",PO30 5AG,448711.00,89155.00,50.700005,-1.311629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
2,100060752724,19458316,"27, ATKINSON DRIVE, NEWPORT, PO30 2LJ",PO30 2LJ,450922.84,89862.56,50.706178,-1.280216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
3,100062422315,19459344,"1 SHEAT COTTAGES, BROOK LANE, CHILLERTON, NEWP...",PO30 3EW,449194.00,84452.00,50.657675,-1.305415,1.000627e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
4,100062423020,19460758,"5 TENNYSON VIEW, ELM LANE, CALBOURNE, NEWPORT,...",PO30 4JS,442474.00,87032.00,50.681402,-1.400181,1.000627e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90771,100060775880,19519235,"ISLAND COTTAGE, TENNYSON ROAD, YARMOUTH, PO41 0PX",PO41 0PX,435657.00,89638.00,50.705293,-1.496415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached
90772,100062448970,19519561,"FORT VICTORIA REPTILARIUM, FORT VICTORIA COUNT...",PO41 0RR,433852.00,89786.00,50.706731,-1.521962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PP,Property Shell
90773,10094234389,55978161,"9, BOULDNOR MEAD, BOULDNOR, YARMOUTH, PO41 0LA",PO41 0LA,436839.66,89727.91,50.706028,-1.479659,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
90774,10094232487,55502866,"LEE BARN, LEE FARM, MAIN ROAD, WELLOW, YARMOUT...",PO41 0SY,438040.00,88601.00,50.695817,-1.462777,1.000334e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached


In [48]:
final_base_plus.columns = final_base_plus.columns.str.title()
final_base_plus.columns = final_base_plus.columns.str.replace("_", "")
final_base_plus = final_base_plus.rename(
    columns={
        "Uprn": "UPRN",
        "Udprn": "UDPRN",
        "Aonb": "AONB",
        "Postcodelocator": "PostcodeLocator",
        "Xcoordinate": "XCoordinate",
        "Ycoordinate": "YCoordinate",
        "Parentuprn": "ParentUPRN",
        "Wardcode": "WardCode",
        "Lsoa11Code": "LSOA11Code",
        "Coa11Code": "COA11Code",
        "Parishcode": "ParishCode",
        "Conservationarea": "ConservationArea",
        "Nationalpark": "NationalPark",
        "Worldheritagesite": "WorldHeritageSite",
        "Classdescription": "ClassDescription",
    },
)

In [49]:
import re


def title_case_address(address):
    # Split by comma
    parts = address.split(", ")

    # Function to check if a part matches the pattern of a postcode
    def is_postcode(part):
        # This is a simple regex pattern for UK postcodes, might need to be adjusted for better accuracy
        return bool(re.match(r"^[A-Z]{1,2}[0-9R][0-9A-Z]? [0-9][ABD-HJLNP-UW-Z]{2}$", part))

    # Title case each part unless it's a postcode
    transformed_parts = [part.upper() if is_postcode(part) else part.title() for part in parts]

    # Join the parts back together
    return ", ".join(transformed_parts)


# Apply the function to the 'address' column
final_base_plus["Address"] = final_base_plus["Address"].apply(title_case_address)

In [50]:
final_base_plus

,UPRN,UDPRN,Address,PostcodeLocator,XCoordinate,YCoordinate,Latitude,Longitude,ParentUPRN,WardCode,LSOA11Code,COA11Code,ParishCode,ConservationArea,NationalPark,WorldHeritageSite,AONB,Class,ClassDescription
0,10003317367,54801750,"Biscoes Solicitors, 143, High Street, Newport",PO30 1TY,450071.00,89235.00,50.700609,-1.292363,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CO01,Office / Work Studio
1,10003323356,19461404,"8, Silver Birch Drive, Newport",PO30 5AG,448711.00,89155.00,50.700005,-1.311629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
2,100060752724,19458316,"27, Atkinson Drive, Newport",PO30 2LJ,450922.84,89862.56,50.706178,-1.280216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
3,100062422315,19459344,"1 Sheat Cottages, Brook Lane, Chillerton, Newport",PO30 3EW,449194.00,84452.00,50.657675,-1.305415,1.000627e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
4,100062423020,19460758,"5 Tennyson View, Elm Lane, Calbourne, Newport",PO30 4JS,442474.00,87032.00,50.681402,-1.400181,1.000627e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90771,100060775880,19519235,"Island Cottage, Tennyson Road, Yarmouth",PO41 0PX,435657.00,89638.00,50.705293,-1.496415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached
90772,100062448970,19519561,"Fort Victoria Reptilarium, Fort Victoria Count...",PO41 0RR,433852.00,89786.00,50.706731,-1.521962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PP,Property Shell
90773,10094234389,55978161,"9, Bouldnor Mead, Bouldnor, Yarmouth",PO41 0LA,436839.66,89727.91,50.706028,-1.479659,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
90774,10094232487,55502866,"Lee Barn, Lee Farm, Main Road, Wellow, Yarmouth",PO41 0SY,438040.00,88601.00,50.695817,-1.462777,1.000334e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached


In [51]:
def remove_postcode_and_title_case(address):
    # Regex pattern for UK postcodes with possible preceding comma and spaces
    postcode_pattern = r"(,\s*)?[A-Z]{1,2}[0-9R][0-9A-Z]? [0-9][ABD-HJLNP-UW-Z]{2}"

    # Remove postcode using regex
    address_without_postcode = re.sub(postcode_pattern, "", address).strip()

    # Remove trailing comma if it exists after removing the postcode
    address_without_postcode = address_without_postcode.rstrip(",")

    # Split by comma
    parts = address_without_postcode.split(", ")

    # Title case each part
    transformed_parts = [part.title() for part in parts]

    # Join the parts back together
    return ", ".join(transformed_parts)


# Apply the function to the 'address' column
final_base_plus["Address"] = final_base_plus["Address"].apply(remove_postcode_and_title_case)

In [52]:
def convert_to_str(val):
    if pd.isna(val) or val == float("inf"):
        return str(val)
    return str(int(val))


final_base_plus["ParentUPRN"] = final_base_plus["ParentUPRN"].apply(convert_to_str)

In [53]:
final_base_plus.to_csv(ROOT / "data" / "interim" / "address_base_plus.csv")

---

## mart_reduced_co2_measures

In [54]:
reduced = pd.read_csv(
    ROOT / "data" / "interim" / "mart_reduced_co2_measures.csv",
    low_memory=False,
)

In [55]:
reduced

,UPRN,UDPRN,AddressId,Address,SAP,EI,KgCO2SAP2012,KgCO22017,KgCO2SAP102,KgCO22025,...,MeasureGroupName,COST,IMPROVEMENT_DESCR_TEXT,CumulativeCost,SAPfollowingthisMeasure,IndividualSAPincrease,CumulativeSAPincrease,AverageHeatLossCoefficient (w/K),MeasureOutcomeId,MeasureOutcomeName
0,2.000029e+11,51630909.0,NaN,"The Boathouse, Ommanney Road, Po41 0Qa",71,71,NaN,NaN,NaN,NaN,...,Low energy lighting for all fixed outlets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.0,Low energy lighting for all fixed outlets
1,2.000029e+11,51630909.0,NaN,"The Boathouse, Ommanney Road, Po41 0Qa",71,71,NaN,NaN,NaN,NaN,...,Solar water heating,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0,Solar water heating
2,2.000029e+11,51630909.0,NaN,"The Boathouse, Ommanney Road, Po41 0Qa",71,71,NaN,NaN,NaN,NaN,...,"Solar photovoltaic panels, 2.5 kWp",NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0,"Solar photovoltaic panels, 2.5 kWp"
3,2.000029e+11,19519008.0,NaN,"1 Alexandra Cottages, Westhill Lane, Norton, P...",67,57,NaN,NaN,NaN,NaN,...,Low energy lighting for all fixed outlets,£65,NaN,NaN,NaN,NaN,NaN,NaN,35.0,Low energy lighting for all fixed outlets
4,2.000029e+11,19519008.0,NaN,"1 Alexandra Cottages, Westhill Lane, Norton, P...",67,57,NaN,NaN,NaN,NaN,...,Upgrade heating controls,£350 - £450,NaN,NaN,NaN,NaN,NaN,NaN,14.0,Upgrade heating controls
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271850,NaN,NaN,NaN,"Unit 2, Thorness Cross Farm, Thorness Lane, Po...",40,34,NaN,NaN,NaN,NaN,...,"Solar photovoltaic panels, 2.5 kWp","£3,500 - £5,500",NaN,NaN,NaN,NaN,NaN,NaN,34.0,"Solar photovoltaic panels, 2.5 kWp"
271851,NaN,NaN,NaN,"Clover Cottage, Stag Lane, Po30 5Ts",31,29,NaN,NaN,NaN,NaN,...,Wind turbine,"£1,500 - £4,000",NaN,NaN,NaN,NaN,NaN,NaN,44.0,Wind turbine
271852,NaN,NaN,NaN,"The Bakehouse, Berryl Farm Cottages, Nettlecom...",59,45,NaN,NaN,NaN,NaN,...,Wind turbine,"£1,500 - £4,000",NaN,NaN,NaN,NaN,NaN,NaN,44.0,Wind turbine
271853,NaN,NaN,NaN,"The Farmhouse Cottage, Appuldurcombe Road, Wro...",26,19,NaN,NaN,NaN,NaN,...,"Solar photovoltaic panels, 2.5 kWp","£9,000 - £14,000",NaN,NaN,NaN,NaN,NaN,NaN,34.0,"Solar photovoltaic panels, 2.5 kWp"


In [57]:
reduced["UPRN"] = reduced["UPRN"].apply(convert_to_str)
reduced["UDPRN"] = reduced["UDPRN"].apply(convert_to_str)

In [58]:
reduced["Address"].apply(title_case_address)

0                    The Boathouse, Ommanney Road, Po41 0Qa
1                    The Boathouse, Ommanney Road, Po41 0Qa
2                    The Boathouse, Ommanney Road, Po41 0Qa
3         1 Alexandra Cottages, Westhill Lane, Norton, P...
4         1 Alexandra Cottages, Westhill Lane, Norton, P...
                                ...                        
271850    Unit 2, Thorness Cross Farm, Thorness Lane, Po...
271851                  Clover Cottage, Stag Lane, Po30 5Ts
271852    The Bakehouse, Berryl Farm Cottages, Nettlecom...
271853    The Farmhouse Cottage, Appuldurcombe Road, Wro...
271854    The Farmhouse Cottage, Appuldurcombe Road, Wro...
Name: Address, Length: 271855, dtype: object

In [59]:
# reduced[reduced["uprn"] == "nan"]
reduced.columns

Index(['UPRN', 'UDPRN', 'AddressId', 'Address', 'SAP', 'EI', 'KgCO2SAP2012',
       'KgCO22017', 'KgCO2SAP102', 'KgCO22025', 'KgCO22030', 'KgCO22038',
       'KgCO22050', 'FuelBills', 'FuelBillsRealistic', 'HeatingCost', 'kWhyr',
       'kWhHeatingDemandm2yr', 'AvHeatLossCoefficientwK', 'TotalFloorArea',
       'ImprovementIndex', 'Category', 'MeasureGroupId', 'MeasureGroupName',
       'COST', 'IMPROVEMENT_DESCR_TEXT', 'CumulativeCost',
       'SAPfollowingthisMeasure', 'IndividualSAPincrease',
       'CumulativeSAPincrease', 'AverageHeatLossCoefficient (w/K)',
       'MeasureOutcomeId', 'MeasureOutcomeName'],
      dtype='object')

In [60]:
import re


def title_case_address(address):
    # Split by comma
    parts = address.split(", ")

    # Function to check if a part matches the pattern of a postcode
    def is_postcode(part):
        # This is a regex pattern for UK postcodes with case-insensitive flag
        return bool(
            re.match(r"^[A-Z]{1,2}[0-9R][0-9A-Z]? [0-9][ABD-HJLNP-UW-Z]{2}$", part, re.IGNORECASE),
        )

    # Title case each part unless it's a postcode
    transformed_parts = [part.upper() if is_postcode(part) else part.title() for part in parts]

    # Join the parts back together
    return ", ".join(transformed_parts)

In [61]:
reduced["Address"] = reduced["Address"].apply(title_case_address)

In [62]:
reduced.columns = reduced.columns.str.title()

In [63]:
reduced = reduced.rename(
    columns={
        "Uprn": "UPRN",
        "Udprn": "UDPRN",
        "Addressid": "AddressID",
        "Sap": "SAP",
        "Ei": "EI",
        "Kgco2Sap2012": "KgCO2SAP2012",
        "Kgco22017": "KgCO22017",
        "Kgco2Sap102": "KgCO2Sap102",
        "Kgco22025": "KgCO22025",
        "Kgco22030": "KgCO22030",
        "Kgco22038": "KgCO22038",
        "Kgco22050": "KgCO22050",
        "Fuelbills": "FuelBills",
        "Fuelbillsrealistic": "FuelBillsRealistic",
        "Heatingcost": "HeatingCost",
        "Kwhyr": "KwhYr",
        "Kwhheatingdemandm2Yr": "KwhHeatingDemandm2Yr",
        "Avheatlosscoefficientwk": "AvHeatLossCoefficientWk",
        "Totalfloorarea": "TotalFloorArea",
        "Measuregroupid": "MeasureGroupID",
        "Measuregroupname": "MeasureGroupName",
        "Improvement_Descr_Text": "ImprovementDescrText",
        "Cumulativecost": "CumulativeCost",
        "Sapfollowingthismeasure": "SAPFollowingThisMeasure",
        "Individualsapincrease": "IndividualSAPIncrease",
        "Cumulativesapincrease": "CumulativeSAPIncrease",
        "Av. Heat Loss Coefficient (W/K)": "AvHeatLossCoefficient(W/K)",
        "Measure Outcome Id": "MeasureOutcomeId",
        "Measure Outcome Name": "MeasureOutcomeName",
    },
)

In [65]:
reduced[reduced["UPRN"] != "nan"].to_csv(
    ROOT / "data" / "interim" / "reduced_co2_measures_2023_09_15_1430.csv",
)

In [66]:
final_base_plus

,UPRN,UDPRN,Address,PostcodeLocator,XCoordinate,YCoordinate,Latitude,Longitude,ParentUPRN,WardCode,LSOA11Code,COA11Code,ParishCode,ConservationArea,NationalPark,WorldHeritageSite,AONB,Class,ClassDescription
0,10003317367,54801750,"Biscoes Solicitors, 143, High Street, Newport",PO30 1TY,450071.00,89235.00,50.700609,-1.292363,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CO01,Office / Work Studio
1,10003323356,19461404,"8, Silver Birch Drive, Newport",PO30 5AG,448711.00,89155.00,50.700005,-1.311629,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
2,100060752724,19458316,"27, Atkinson Drive, Newport",PO30 2LJ,450922.84,89862.56,50.706178,-1.280216,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
3,100062422315,19459344,"1 Sheat Cottages, Brook Lane, Chillerton, Newport",PO30 3EW,449194.00,84452.00,50.657675,-1.305415,100062705590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
4,100062423020,19460758,"5 Tennyson View, Elm Lane, Calbourne, Newport",PO30 4JS,442474.00,87032.00,50.681402,-1.400181,100062705868,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90771,100060775880,19519235,"Island Cottage, Tennyson Road, Yarmouth",PO41 0PX,435657.00,89638.00,50.705293,-1.496415,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached
90772,100062448970,19519561,"Fort Victoria Reptilarium, Fort Victoria Count...",PO41 0RR,433852.00,89786.00,50.706731,-1.521962,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PP,Property Shell
90773,10094234389,55978161,"9, Bouldnor Mead, Bouldnor, Yarmouth",PO41 0LA,436839.66,89727.91,50.706028,-1.479659,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD03,Semi-Detached
90774,10094232487,55502866,"Lee Barn, Lee Farm, Main Road, Wellow, Yarmouth",PO41 0SY,438040.00,88601.00,50.695817,-1.462777,10003340559,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RD02,Detached
